---
title: F1 Track Visualization
format: 
    html:
        embed-resources: true
        theme: dark
---


# F1 Track Visualization Development Process

## Initial Approach
- Created basic track visualization using telemetry data
- Implemented color-coded track based on telemetry variables (Speed, Throttle, etc.)
- Added static points for driver positions

### Problems Faced
1. Static visualization didn't show race progression
2. No comparison between different drivers
3. Limited interactivity

## First Iteration: Adding Animation
- Implemented matplotlib animation for driver positions
- Added comparison with fastest lap
- Created interactive dropdowns for:
  - Race selection
  - Driver selection
  - Lap selection
  - Telemetry variable selection

### Problems Faced
1. Animation size exceeded Jupyter's limit (20MB)
2. Points disappeared after initial animation
3. Cache size grew uncontrollably
4. Animation continued after lap completion

## Second Iteration: Optimization
- Reduced animation size through:
  - Lower DPI (80)
  - Reduced frame rate
  - Sampled telemetry data
- Added cache management system
- Fixed point visibility issues

### Problems Faced
1. Widget updates broke animation
2. Index out of bounds errors
3. Multiple plots appearing
4. Memory leaks

## Third Iteration: Robust Implementation
- Implemented proper cleanup between updates
- Added error handling
- Optimized animation parameters:
  - 10-second duration
  - 30 FPS
  - Automatic sample rate calculation
- Added cache size monitoring

### Final Optimizations
1. Animation Parameters:
   - `desired_duration = 10 seconds`
   - `target_fps = 30`
   - `total_frames = 300`
   - `interval = 33.3ms`
   - `dpi = 80`

2. Memory Management:
   - Cache size monitoring
   - Automatic cache clearing
   - Figure cleanup
   - Reduced data sampling

3. Error Handling:
   - Index validation
   - Try-except blocks
   - Proper widget state management

## Current Features
1. Interactive Track Visualization:
   - Color-coded telemetry data
   - Animated driver positions
   - Fastest lap comparison
   - Multiple telemetry variables

2. User Controls:
   - Race selection
   - Driver selection
   - Lap selection (Fastest/Slowest)
   - Telemetry variable selection

3. Performance Optimizations:
   - Memory-efficient animation
   - Smooth playback
   - Automatic cache management
   - Error-resistant operation

## Lessons Learned
1. Animation Optimization:
   - Balance between quality and performance
   - Importance of proper sampling
   - Frame rate management

2. Memory Management:
   - Cache control importance
   - Figure cleanup necessity
   - Data sampling strategies

3. Error Handling:
   - Widget state management
   - Index validation
   - Proper exception handling

4. User Experience:
   - Smooth transitions
   - Responsive controls
   - Clear visual feedback

## Future Improvements
1. Additional Features:
   - More lap selection options
   - Multiple driver comparisons
   - Sector timing information
   - Race position overlay

2. Performance:
   - Further optimization of animation size
   - Improved caching strategies
   - Better data sampling methods

3. User Interface:
   - Additional control options
   - Better visual feedback
   - More telemetry variables

### Races to keep:

**Monaco Grand Prix**

- Narrow street circuit with minimal overtaking opportunities
- Emphasis on qualifying — grid position is critical
- Common use of early pit stops for undercut strategies
- High chance of safety car disruptions
- Strategy prioritizes track position over raw pace

**Italian Grand Prix**

- High-speed track with long straights and minimal corners
- Teams run low-downforce setups for top speed
- Slipstreaming is key during qualifying
- Tire wear is low — allows for aggressive stint strategies
- Fuel efficiency and power unit performance are major factors

**Singapore Grand Prix**

- Physically demanding race due to heat, humidity, and race length
- Tight street circuit with high tire degradation
- Frequent safety cars demand adaptable strategy
- High likelihood of full-race duration reaching 2-hour time limit
- Focus on managing driver fatigue and fuel/tire conservation

**Belgian Grand Prix**

- Long, flowing track with major elevation changes
- Unpredictable weather — can vary across different sectors
- Strategy must account for sudden rain or mixed conditions
- Tire compound choice is often reactive
- Overtaking-friendly but demands precise timing for stops

**United States Grand Prix**

- Mix of high-speed corners, hairpins, and straights
- Requires a well-balanced aerodynamic setup
- Medium-to-high tire degradation — often leads to 2-stop races
- Wide track layout encourages strategic overtaking
- Pit stop timing and tire strategy play a major role


In [1]:
import pandas as pd
pd.set_option('display.max_columns', None)
import altair as alt
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
from matplotlib import colormaps
import matplotlib as mpl
import numpy as np

import fastf1 as ff1
import fastf1.plotting

import ipywidgets as widgets
from IPython.display import display, clear_output


import logging
logging.getLogger('fastf1').setLevel(logging.ERROR)
import warnings
warnings.filterwarnings('ignore')


In [2]:
# TODO: get rid of this function, change to 2024 only
def get_schedule(year: int):
    """
    Returns the list of events for a given year (2018-2024)
    """
    return pd.DataFrame(ff1.get_event_schedule(year))

# TODO: Create dropdown of event_names from get_schedule(year)
def load_race_session(year: int, event_name: str):
    """
    Returns the details of the given race event (Races only)
    """
    session = ff1.get_session(year, event_name, 'Race')
    session.load()
    return session

# TODO: Allow user to select driver name, which then maps to driver code
def get_driver_laps(session, driver_code: str):
    """
    Returns the laps for a given driver in a given session
    """
    laps = session.laps.pick_driver(driver_code)
    return laps

# TODO: Only take every 10th lap + fastest lap for driver
def get_lap_telemetry(lap):
    """
    Returns the telemetry for a given lap
    """
    tel = lap.get_car_data().add_distance()
    return tel

# TODO: get separate telemetry data for the fastest lap overall for race

## Interaction Logic


- User selects an event from 2024
    - Get telemetry data for the fastest overall lap for the race
- User selects a driver
    - Get telemetry data for their fastest lap
- Build out a single track visualization - Animate track position for each driver over time
    - Background 

In [4]:
import ipywidgets as widgets
from matplotlib.colors import ListedColormap
import matplotlib.animation as animation
from IPython.display import HTML
from pathlib import Path
import shutil

### HELPERS TO MANAGE CACHE ###
def get_cache_size_mb():
    cache_size = ff1.Cache.get_cache_info()
    if cache_size[1] is None:
        return 0
    # Return second object in tuple and convert to MB
    return cache_size[1] / (1024 * 1024)

def manage_cache(max_size_mb = 500):
    current_size = get_cache_size_mb()
    if current_size > max_size_mb:
        print(f"Cache size is {current_size} MB, which is greater than the maximum of {max_size_mb} MB. Clearing cache...")
        ff1.Cache.clear_cache()
        print(f"Cache cleared. Current size is {get_cache_size_mb()} MB.")


%matplotlib widget




race_dropdown = widgets.Dropdown(description='Race:')
driver_dropdown = widgets.Dropdown(description='Driver:')
lap_dropdown = widgets.Dropdown(description='Lap:')
telemetry_dropdown = widgets.Dropdown(description='Variable:', options=['Speed', 'Throttle', 'nGear', 'Brake', 'RPM'])




output = widgets.Output()

def show_telemetry(change):
    with output:
        output.clear_output(wait=True)
        plt.close('all')
        
        try:
            year = 2024
            race = race_dropdown.value
            driver = driver_dropdown.value
            lap_num = lap_dropdown.value
            telemetry_var = telemetry_dropdown.value

            manage_cache()
            
            session = load_race_session(year, race)
            fastest_lap_data = session.laps.pick_fastest()
            fastest_driver = fastest_lap_data['Driver']
            fastest_tel = fastest_lap_data.get_telemetry()
            
            driver_laps = get_driver_laps(session, driver)
            selected_lap = driver_laps.loc[driver_laps['LapNumber'] == lap_num].iloc[0]
            selected_tel = selected_lap.get_telemetry()

            def create_static_track():
                x = selected_tel['X'].to_numpy()
                y = selected_tel['Y'].to_numpy()
                color = selected_tel[telemetry_var].to_numpy()

                points = np.array([x, y]).T.reshape(-1, 1, 2)
                segments = np.concatenate([points[:-1], points[1:]], axis=1)

                fig, ax = plt.subplots(figsize=(12, 7))

                jet_map = plt.get_cmap('jet')
                n_gears = 8
                gear_colors = jet_map(np.linspace(0.1, 0.9, n_gears))

                cmap_dict = {
                    'Speed': 'plasma',
                    'Throttle': 'viridis',
                    'nGear': ListedColormap(gear_colors),
                    'Brake': ListedColormap(['gray', 'red']),
                    'RPM': 'inferno'
                }

                norm = plt.Normalize(np.nanmin(color), np.nanmax(color))
                lc = LineCollection(segments, cmap=cmap_dict[telemetry_var], norm=norm)
                lc.set_array(color)
                lc.set_linewidth(4)
                
                line = ax.add_collection(lc)
                ax.plot(x, y, color='lightgray', linewidth=1, alpha=0.5)
                
                cbar = plt.colorbar(line, ax=ax)
                units = {
                    'Speed': 'km/h',
                    'Throttle': '%',
                    'nGear': 'Gear',
                    'Brake': 'On/Off',
                    'RPM': 'RPM'
                }
                cbar.set_label(f'{telemetry_var} ({units[telemetry_var]})', fontsize=10)
                
                selected_point, = ax.plot([selected_tel['X'].iloc[0]], [selected_tel['Y'].iloc[0]], 
                                        'ko', markersize=10, label=f'{driver}', zorder=10)
                fastest_point, = ax.plot([fastest_tel['X'].iloc[0]], [fastest_tel['Y'].iloc[0]], 
                                       'o', color='gold', markersize=10, 
                                       label=f'{fastest_driver} (Fastest)', zorder=10)
                
                ax.set_xlim(min(selected_tel['X'].min(), fastest_tel['X'].min()) - 100,
                           max(selected_tel['X'].max(), fastest_tel['X'].max()) + 100)
                ax.set_ylim(min(selected_tel['Y'].min(), fastest_tel['Y'].min()) - 100,
                           max(selected_tel['Y'].max(), fastest_tel['Y'].max()) + 100)
                
                ax.legend(loc='upper right')
                ax.axis('equal')
                ax.axis('off')
                ax.set_title(f'{driver} vs Fastest Lap ({fastest_driver}) - {race} {year}', pad=20)
                plt.tight_layout()
                
                return fig, ax, selected_point, fastest_point

            fig, ax, selected_point, fastest_point = create_static_track()
            
            # Get calid indices for both telemetry datasets
            selected_valid = selected_tel[['X', 'Y']].dropna().index.to_numpy()
            fastest_valid = fastest_tel[['X', 'Y']].dropna().index.to_numpy()

            # Calculate params for 10-second animation
            desired_duration = 10
            target_fps = 15
            total_frames = desired_duration * target_fps

            # calculate sample rate based on data length
            min_length = min(len(selected_valid), len(fastest_valid))
            sample_rate = max(min_length // total_frames, 1)

            # create evenly spaced indices
            frame_indices = np.linspace(0, min_length - 1, total_frames).astype(int)

            def animate(frame_idx):
                try:
                    # get indices for this frame
                    idx = frame_indices[frame_idx]
                    selected_idx = selected_valid[idx]
                    fastest_idx = fastest_valid[idx]

                    # update both points
                    selected_point.set_data([selected_tel['X'].iloc[selected_idx]], 
                                          [selected_tel['Y'].iloc[selected_idx]])
                    fastest_point.set_data([fastest_tel['X'].iloc[fastest_idx]], 
                                         [fastest_tel['Y'].iloc[fastest_idx]])
                    return selected_point, fastest_point
                except IndexError:
                    print(f"Index error at frame {frame_idx}")
                    return selected_point, fastest_point

            anim = animation.FuncAnimation(
                fig,
                animate,
                frames=len(frame_indices),
                interval=1000 * target_fps,
                blit=True
            )

            plt.rcParams['figure.dpi'] = 80
            html_animation = anim.to_jshtml(
                fps=target_fps,
                default_mode='once',
                embed_frames=True
            )
            
            plt.close(fig)
            display(HTML(html_animation))
                
        except Exception as e:
            print(f"Error occurred: {e}")
            import traceback
            traceback.print_exc()

def update_races(change=None):
    try:
        manage_cache()
        year = 2024
        schedule = get_schedule(year)

        selected_races = [
            'Monaco Grand Prix',
            'Italian Grand Prix',
            'Singapore Grand Prix',
            'Belgian Grand Prix',
            'United States Grand Prix'
        ]

        filtered_schedule = schedule[schedule['EventName'].isin(selected_races)]
        race_dropdown.options = filtered_schedule['EventName'].tolist()
        race_dropdown.value = race_dropdown.options[0]
        update_drivers(None)
    except Exception as e:
        print(f"Error in update_races: {e}")

def update_drivers(change):
    try:
        manage_cache()
        year = 2024
        race = race_dropdown.value
        session = load_race_session(year, race)
        results = session.results
        
        driver_dropdown.options = results['Abbreviation'].tolist()
        driver_dropdown.value = driver_dropdown.options[0]
        
        output.clear_output(wait=True)
        update_laps(None)
    except Exception as e:
        print(f"Error in update_drivers: {e}")

def update_laps(change):
    try:
        manage_cache()
        year = 2024
        race = race_dropdown.value
        driver = driver_dropdown.value
        session = load_race_session(year, race)
        driver_laps = get_driver_laps(session, driver)
        
        completed_laps = driver_laps[
            (driver_laps['IsAccurate'] == True) & 
            (driver_laps['Time'].notna())
        ]
        
        fastest_lap = completed_laps.loc[completed_laps['LapTime'].idxmin()]
        slowest_lap = completed_laps.loc[completed_laps['LapTime'].idxmax()]
        
        lap_dropdown.options = [
            (f"Lap {fastest_lap['LapNumber']} (Fastest)", fastest_lap['LapNumber']),
            (f"Lap {slowest_lap['LapNumber']} (Slowest)", slowest_lap['LapNumber'])
        ]
        lap_dropdown.value = fastest_lap['LapNumber']
        
        output.clear_output(wait=True)
        show_telemetry(None)
    except Exception as e:
        print(f"Error in update_laps: {e}")

# Initialize observers
race_dropdown.observe(update_drivers, names='value')
driver_dropdown.observe(update_laps, names='value')
lap_dropdown.observe(show_telemetry, names='value')
telemetry_dropdown.observe(show_telemetry, names='value')

# Initial setup

# Create and display UI
ui = widgets.VBox([
    widgets.HBox([race_dropdown, driver_dropdown]),
    widgets.HBox([lap_dropdown, telemetry_dropdown]),
    output
])

update_races(None)
#display(ui)

In [ ]:
display(ui)